# Exp No 1: StyleGAN FFHQ projection attempt

In [ ]:
import os

!pip install ninja imageio imageio-ffmpeg -q

if not os.path.exists(stylegan_dir := "/content/stylegan2-ada-pytorch"):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git {stylegan_dir}

In [ ]:
import gdown

if not os.path.exists(model_path := "/content/ffhq.pkl"):
    gdown.download(
        "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl",
        model_path, quiet=False
    )

### 3. Mount drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := "/content/gh0st-in-the-l00p"):
    !git -C {repo_dir} pull
else:
    !git clone https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}

### 4. Settings

In [ ]:
drive_root = "/content/drive/MyDrive/Gh0st in the Loop"

# Prepared images group to invert
input_group = f"{drive_root}/dataset/prepared/rhinestones"

# Output dir for projections and videos
output_dir = f"{drive_root}/outputs"

# How many images to invert from the group
n_images = 3

# Projection steps (500 = good balance, 1000 for finals)
projection_steps = 500

# Interpolation frames between each pair
interp_steps = 30

# Video FPS
fps = 24

### 5. Load model

In [ ]:
import sys
sys.path.insert(0, stylegan_dir)

import torch_utils
import dnnlib
import torch
import pickle
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

with open(model_path, "rb") as f:
    G = pickle.load(f)["G_ema"].to(device)

print("Model loaded.")

### 6. Load and preview input images

In [ ]:
from PIL import Image, ImageOps
from pathlib import Path
import numpy as np

image_paths = sorted([
    p for p in Path(input_group).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
])[:n_images]

print(f"Using {len(image_paths)} images from {Path(input_group).name}:")
grid = Image.new("RGB", (len(image_paths) * 256, 256))
for i, p in enumerate(image_paths):
    img = ImageOps.exif_transpose(Image.open(p)).resize((256, 256))
    grid.paste(img, (i * 256, 0))
display(grid)

### 7. Project images into latent space

In [ ]:
from pathlib import Path

proj_dir = f"{output_dir}/projections"
os.makedirs(proj_dir, exist_ok=True)

projected = []

for i, path in enumerate(image_paths):
    out_path = f"{proj_dir}/{path.stem}"
    print(f"\n[{i+1}/{len(image_paths)}] Projecting {path.name}...")

    !python {stylegan_dir}/projector.py \
        --outdir="{out_path}" \
        --target="{str(path)}" \
        --network={model_path} \
        --num-steps={projection_steps} \
        > /tmp/proj_log.txt 2>&1

    !tail -5 /tmp/proj_log.txt

    if os.path.exists(w_path := f"{out_path}/projected_w.npz"):
        projected.append((path.stem, np.load(w_path)["w"]))
        print(f"  ✅ {w_path}")
    else:
        print(f"  ❌ Failed for {path.name}")

print(f"\nProjected {len(projected)}/{len(image_paths)} images.")

### 8. Preview projections vs originals

In [ ]:
def w_to_image(w_np):
    w_tensor = torch.tensor(w_np).to(device)
    img = G.synthesis(w_tensor, noise_mode="const")
    img = (img.permute(0,2,3,1) * 127.5 + 128).clamp(0,255).to(torch.uint8)
    return Image.fromarray(img[0].cpu().numpy())

grid = Image.new("RGB", (len(projected) * 512, 256))
for i, (name, w) in enumerate(projected):
    orig = ImageOps.exif_transpose(
        Image.open(next(p for p in image_paths if p.stem == name))
    ).resize((256, 256))
    grid.paste(orig, (i * 512, 0))
    grid.paste(w_to_image(w).resize((256, 256)), (i * 512 + 256, 0))

display(grid)
print("original | projected  for each image")

### 9. Interpolate and render video

In [ ]:
import imageio

os.makedirs(output_dir, exist_ok=True)
video_path = f"{output_dir}/interpolation.mp4"

with imageio.get_writer(video_path, fps=fps) as writer:
    for i in range(len(projected)):
        name_a, w_a = projected[i]
        name_b, w_b = projected[(i + 1) % len(projected)]
        print(f"  {name_a} → {name_b}")

        for t in np.linspace(0, 1, interp_steps):
            frame = w_to_image((1 - t) * w_a + t * w_b)
            writer.append_data(np.array(frame))

print(f"\n✅ Video saved to {video_path}")

### 10. Preview video inline

In [ ]:
from IPython.display import HTML
from base64 import b64encode

data_url = "data:video/mp4;base64," + b64encode(open(video_path, "rb").read()).decode()
display(HTML(f'<video width=512 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))